# Practical 1: Handwritten Digit Classification using a Basic CNN (MNIST)

**Aim:** To design, train and evaluate a Convolutional Neural Network (CNN) that classifies 28x28 grayscale images of handwritten digits (0-9) from the MNIST dataset.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Test accuracy ~99.0 - 99.3%.; Use sparse_categorical_crossentropy with integer labels; normalise pixels to [0,1].

## Cell 1 - Import libraries

**What it does:**
- Imports NumPy (arrays), Matplotlib (graphs), TensorFlow/Keras (building the network) and scikit-learn (evaluation tools).
- `layers` gives Conv2D, MaxPooling2D, Dense etc.; `confusion_matrix` and `classification_report` are used after training.
- Printing the version confirms TensorFlow is installed correctly.

*Expected output (illustrative):* TensorFlow version: 2.x.x  (your installed version)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## Cell 2 - Load the MNIST dataset

**What it does:**
- `load_data()` returns two tuples: training images/labels and test images/labels. Labels are integers 0-9 (not one-hot).
- `x_train` holds the images, `y_train` the digit each image shows. The shapes check that all data was loaded.

*Expected output (illustrative):* x_train: (60000, 28, 28) y_train: (60000,)
x_test : (10000, 28, 28) y_test : (10000,)
Pixel range: 0 to 255

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test :", x_test.shape, "y_test :", y_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

## Cell 3 - Preprocess the data

**What it does:**
- Dividing by 255 normalises pixels to [0, 1]. Neural networks train faster and more stably on small values. Forgetting this is a classic cause of poor accuracy.
- `np.expand_dims(..., -1)` adds the channel axis: (28, 28) becomes (28, 28, 1). Conv2D expects 4-D input (batch, height, width, channels).
- Labels stay as integers, so we will use `sparse_categorical_crossentropy`.

*Expected output (illustrative):* x_train: (60000, 28, 28, 1) x_test: (10000, 28, 28, 1)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

class_names = [str(i) for i in range(10)]
print("x_train:", x_train.shape, "x_test:", x_test.shape)

## Cell 4 - Visualise sample images

**What it does:**
- Shows the first 10 training digits with their labels so you can confirm the data looks right.
- `squeeze()` removes the channel axis so `imshow` can draw the 28x28 image.

*Expected output (illustrative):* A 2 x 5 grid of handwritten digits with their labels (5, 0, 4, 1, 9, 2, 1, 3, 1, 4 for the standard MNIST training set).

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title("Label: " + str(y_train[i]))
    plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 5 - Build the CNN model

**What it does:**
- `Sequential` stacks layers one after another.
- `Conv2D(32, (3,3))`: 32 filters of size 3x3 detect edges and strokes. `Conv2D(64, ...)` learns more complex shapes.
- `MaxPooling2D((2,2))` keeps the maximum of each 2x2 block, halving height and width and adding a little shift-tolerance.
- `Flatten` turns the 5x5x64 maps into a 1600-long vector. `Dense(128)` combines features. `Dropout(0.5)` randomly switches off half the neurons during training to reduce overfitting.
- `Dense(10, softmax)` outputs 10 probabilities that sum to 1 - one per digit.

*Expected output (illustrative):* model.summary() lists: conv2d (26,26,32) 320 params; max_pooling2d (13,13,32); conv2d_1 (11,11,64) 18,496; max_pooling2d_1 (5,5,64); flatten (1600); dense (128) 204,928; dropout; dense_1 (10) 1,290.  Total params: 225,034.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])

model.summary()

## Cell 6 - Compile the model

**What it does:**
- **Optimizer Adam** adapts the learning rate per parameter (momentum + RMSprop); lr = 0.001 is the standard start.
- **Loss** `sparse_categorical_crossentropy` is used because labels are integers 0-9. (If labels were one-hot you would use `categorical_crossentropy`.)
- `metrics=['accuracy']` reports accuracy every epoch.

*Expected output (illustrative):* No output (the model is configured).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Cell 7 - Train the model

**What it does:**
- `epochs=10`: the whole training set is seen 10 times. `batch_size=128`: weights are updated after every 128 images.
- `validation_split=0.1` keeps 10% of the training data aside (never trained on) to measure generalisation each epoch.
- `history` stores loss/accuracy per epoch for plotting.

*Expected output (illustrative):* 10 lines like  'Epoch 1/10 ... accuracy: 0.9x - loss: 0.xx - val_accuracy: 0.98xx'. Training accuracy rises and loss falls each epoch (exact numbers differ per run).

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

## Cell 8 - Plot accuracy and loss curves

**What it does:**
- Left graph: accuracy rising towards ~0.99. Right graph: loss falling.
- If validation loss starts rising while training loss keeps falling, the model is overfitting.

*Expected output (illustrative):* Two line graphs (train vs validation). Expected shape: accuracy curves climb and flatten near 0.98-0.99; loss curves drop quickly then flatten.

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 9 - Evaluate on the test set

**What it does:**
- `evaluate` runs the trained model on the 10,000 unseen test images and returns the loss and accuracy.
- This is the final, honest performance figure.

*Expected output (illustrative):* Test accuracy about 0.990 - 0.993 (your PDF's expected range: 99.0 - 99.3%).

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Test loss    :", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))

## Cell 10 - Confusion matrix and classification report

**What it does:**
- `model.predict` returns 10 probabilities per image; `np.argmax` picks the most likely digit.
- The confusion matrix shows true labels (rows) vs predicted labels (columns). The diagonal = correct predictions.
- `classification_report` prints precision, recall and F1-score per digit.

*Expected output (illustrative):* A 10 x 10 heat-map with a strong diagonal, then a table of precision/recall/F1 (about 0.99 for every digit). Most-confused pairs (from your PDF): 4<->9, 3<->5, 7<->2.

In [ ]:
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

## Cell 11 - Predict and display individual test images

**What it does:**
- Picks 10 random test images and shows true vs predicted label (green = correct, red = wrong).
- `model.save` stores the trained model so you can reload it later with `keras.models.load_model`.

*Expected output (illustrative):* 10 digit images titled e.g. 'True 7 / Pred 7' (green). Almost all will be green. A file mnist_cnn.keras is created.

In [ ]:
indices = np.random.choice(len(x_test), 10, replace=False)

plt.figure(figsize=(12, 4))
for pos, idx in enumerate(indices):
    plt.subplot(2, 5, pos + 1)
    plt.imshow(x_test[idx].squeeze(), cmap="gray")
    color = "green" if y_pred[idx] == y_test[idx] else "red"
    plt.title("True " + str(y_test[idx]) + " / Pred " + str(y_pred[idx]), color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

model.save("mnist_cnn.keras")

## Result / Conclusion

A CNN with two Conv-Pool blocks and a dense classifier was trained on MNIST and reached about 99% test accuracy, showing that convolutional layers extract digit features effectively.